In [1]:
import sys
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path
import kagglehub

c:\Users\BIT\Desktop\Deep_Generative_Modelling\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
current_dir = Path().cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("done")

done


In [4]:
data_path = project_root / "data"

In [ ]:
# cache_path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")
# print("Path to dataset files:", cache_path)

# for item in Path(cache_path).iterdir():
#     shutil.copy2(item, data_path/item.name)
    
# print("Data Saved")

100%|██████████| 195M/195M [00:41<00:00, 4.98MB/s] 

Extracting files...


Path to dataset files: C:\Users\BIT\.cache\kagglehub\datasets\grouplens\movielens-20m-dataset\versions\1
Data Saved


In [ ]:
# for file in data_path.iterdir():
#     print(str(file))

c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\Exam_Score_Prediction.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\genome_scores.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\genome_tags.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\link.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\movie.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\rating.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\tag.csv


In [5]:
movie_df = pd.read_csv(str(data_path / "movie.csv"))
rating_df = pd.read_csv(str(data_path / "rating.csv"))

In [6]:
movie_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [8]:
movie_df.shape, rating_df.shape

((27278, 3), (20000263, 4))

In [9]:
rating_df = rating_df.iloc[0:1000000]

In [10]:
len(rating_df["userId"].unique())

6743

In [11]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [12]:
rating_df.isna().all()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [13]:
movie_df.isna().all()

movieId    False
title      False
genres     False
dtype: bool

In [14]:
# Let's create a pivot table to better understand the data
pivot_ratings_df = rating_df.pivot(index="userId", columns="movieId", values="rating")

In [15]:
pivot_ratings_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
pivot_ratings_df.shape

(6743, 13950)

In [17]:
pivot_ratings_df_cy = pivot_ratings_df.copy()

In [38]:
# def binaryconvertor(rating_df):
#     if rating_df <= 2:
#         return 0
#     else:
#         return 1

In [39]:
# # pivot_ratings_df.applymap(lambda x: binaryconvertor(x) if pd.notna(x) else x)
# # let's use numpy
# mask = pivot_ratings_df.notna()
# pivot_ratings_df[mask] = pivot_ratings_df[mask].apply(binaryconvertor)

In [18]:
pivot_ratings_df_cy = pd.DataFrame(
    np.where(pivot_ratings_df_cy.isna(), -1, (pivot_ratings_df_cy > 2).astype(int)),
    index=pivot_ratings_df_cy.index,
    columns=pivot_ratings_df_cy.columns
)

In [19]:
pivot_ratings_df_cy.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
2,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
3,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
4,-1,-1,-1,-1,-1,1,-1,-1,-1,1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
5,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
6,1,-1,1,-1,-1,-1,1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
7,-1,-1,1,-1,-1,-1,1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
8,1,-1,1,-1,-1,1,-1,-1,-1,1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
9,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [20]:
pivot_ratings_df_cy.shape

(6743, 13950)

In [21]:
pivot_ratings_df_cy.index

Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
       ...
       6734, 6735, 6736, 6737, 6738, 6739, 6740, 6741, 6742, 6743],
      dtype='int64', name='userId', length=6743)

In [22]:
pivot_ratings_df.isna().all()

movieId
1         False
2         False
3         False
4         False
5         False
          ...  
130073    False
130219    False
130462    False
130490    False
130642    False
Length: 13950, dtype: bool

Starting to build the RBM

In [23]:
NV = pivot_ratings_df_cy.shape[1]
NH = 100
BATCH_SIZE = 100

In [24]:
weigths = np.random.standard_normal(size=(NV, NH))
a = np.zeros((1, NH))
b = np.zeros((1, NV))

In [25]:
def sigmoid(mat):
    return 1/(1 + (np.exp(-mat)))

In [26]:
# testing it
sample = np.ones((3,4))
sigmoid(sample)

array([[0.73105858, 0.73105858, 0.73105858, 0.73105858],
       [0.73105858, 0.73105858, 0.73105858, 0.73105858],
       [0.73105858, 0.73105858, 0.73105858, 0.73105858]])

In [27]:
def sample_h(sample, weigths, biases):
    activation = sample @ weigths + biases
    prob_h = sigmoid(activation)
    binary = np.random.binomial(n=1, p=prob_h)
    return prob_h, binary 

In [28]:
def sample_v(input, weigths, baises):
    activation = input @ weigths.T + baises
    prob_v = sigmoid(activation)
    binary = np.random.binomial(n=1, p=prob_v)
    return prob_v, binary

In [29]:
# We will start training
nb_epoch = 10
learning_rate = 0.01

In [30]:
def contrastive_divergence(v0: np.array, weigths: np.array, a: np.array, b: np.array, mask: np.array, learning_rate: float):
    """Performs onr step of CD

    Args:
        v0 (np.array): batch of real data
        weigths (np.array): weigths of the RBM
        a (np.array): biases for hidden layer
        b (np.array): biases for visual layer
        mask (np.array): mask array for missing values
        learning_rate (float): learning_rate
        
    Returns:
        updated_weigths, updated_a, updated_b, reconstruction_error
    """
    
    # positive phase
    prob_h0, h0_binary = sample_h(v0, weigths, a)
    
    # negative phase
    prob_v1, v1_sampled = sample_v(h0_binary, weigths, b)
    v1_final = np.where(mask, v1_sampled, v0)
    
    # Sample hidden from reconstructed visible
    prob_h1, h1_binary = sample_h(v1_final, weigths, a)
    
    # Computing gradients
    positive_grad = (v0.T @ prob_h0) / v0.shape[0]
    negative_grad = (v1_final.T @ prob_h1) / v0.shape[0]
    
    # Updating the weigths and biases
    weigths_update = learning_rate * (positive_grad - negative_grad)
    new_weigths = weigths + weigths_update
    
    a_update = learning_rate * np.mean(prob_h0 - prob_h1, axis=0)
    new_a = a + a_update
    
    difference = (v0 - v1_final) * mask
    b_update = learning_rate * np.sum(difference, axis=0) / np.sum(mask, axis=0, keepdims=True)
    new_b = b + b_update
    
    # Calculate reconstruction error
    error = np.sum((v0 - v1_final) ** 2 * mask) / np.sum(mask)
    
    return new_weigths, new_a, new_b, error

In [32]:
indexes = np.array(pivot_ratings_df_cy.index)
np.random.shuffle(indexes)

In [33]:
indexes

array([4373,  569, 2562, ..., 1746, 2934, 2731], shape=(6743,))

In [37]:
def train_rbm(data: np.array, mask: np.array, weigths: np.array, a: np.array, b: np.array, epochs:int, batch_size: int, learning_rate: float):
    """Function to train the RBM.

    Args:
        data (np.array): Training Data
        mask (np.array): mask for missing values
        weigths (np.array): weighths of the RBM
        a (np.array): biases for hidden layer
        b (np.array): biases for visual layer
        epochs (int): No of epochs to train
        batch_size (int): batch size
        learning_rate (float): learning rate
    """
    
    num_samples = data.shape[0]
    num_batches = num_samples // batch_size
    error_history = []
    
    for epoch in tqdm(range(epochs)):
        epoch_error = 0
        indexes = np.arange(num_samples)
        np.random.shuffle(indexes)
        
        shuffled_data = data[indexes]
        shuffled_mask = mask[indexes]
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = start_idx + batch_size
            
            batch_data = shuffled_data[start_idx : end_idx]
            batch_mask = shuffled_mask[start_idx : end_idx]
            
            # perform CD
            weigths, a, b, batch_error = contrastive_divergence(
                batch_data, weigths, a, b, batch_mask, learning_rate
            )
            
            epoch_error += batch_error
            
        avg_error = epoch_error / num_batches
        error_history.append(avg_error)
        
        # scheduler
        if epoch % 3 == 0 and epoch != 0:
            learning_rate /= 10
        
        # Print progress
        if epoch % 1 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Error: {avg_error:.4f}")
        
    return weigths, a, b, error_history

In [38]:
mask = pivot_ratings_df_cy < 0
# mask.loc[[1,2,3]]

In [39]:
train_rbm(
    data=pivot_ratings_df_cy.values,
    mask=mask.values,
    weigths=weigths,
    a=a,
    b=b,
    epochs=10,
    batch_size=BATCH_SIZE,
    learning_rate=0.01
)

  0%|          | 0/10 [00:00<?, ?it/s]C:\Users\BIT\AppData\Local\Temp\ipykernel_23424\3716844040.py:2: RuntimeWarning: overflow encountered in exp
  return 1/(1 + (np.exp(-mat)))
 10%|█         | 1/10 [00:23<03:31, 23.49s/it]

Epoch 1/10, Error: 1.1314


 20%|██        | 2/10 [00:43<02:50, 21.30s/it]

Epoch 2/10, Error: 1.0000


 30%|███       | 3/10 [01:02<02:22, 20.37s/it]

...

 90%|█████████ | 9/10 [02:57<00:19, 19.37s/it]

Epoch 9/10, Error: 1.0000


100%|██████████| 10/10 [03:16<00:00, 19.68s/it]

Epoch 10/10, Error: 1.0000


(array([[-0.40886339, -0.57862707,  0.74853704, ...,  0.76233423,
         -1.57314215, -0.72136822],
        [-1.76922819, -2.58410989, -0.97602201, ..., -2.05681053,
         -2.04274946, -1.39610829],
        [-1.79159626, -1.82987021, -2.68180129, ..., -3.90419569,
         -1.73072049, -3.16147458],
        ...,
        [-2.18540761, -0.28055814, -2.36030781, ..., -1.75786098,
         -3.78166958, -1.85665908],
        [-1.47816101, -2.69361587, -1.66218936, ..., -2.64984236,
         -3.5145726 , -1.65911366],
        [-1.99510408, -2.46425142, -2.72440076, ..., -4.16825519,
         -3.20658899, -2.96554603]], shape=(13950, 100)),
 array([[2.73650747, 2.7198599 , 2.70746438, 2.78123014, 2.77492785,
         2.45383811, 2.74451167, 2.66841301, 2.79639585, 2.7278743 ,
         2.76167705, 2.71962915, 2.75383406, 2.79363019, 2.76129846,
         2.7585228 , 2.71448918, 2.71392075, 2.75750248, 2.76063206,
         2.76918883, 2.75381844, 2.80431862, 2.78090867, 2.76522528,
        